## PLM-based Classification of Direct and Indirect Speech Acts in Korean

본 노트북은 한국어 간접화행의 분류 가능성을 탐색하기 위해
사전학습 언어모델(PLM)을 파인튜닝한 실험 과정을 정리한다.

본 모델은 자동 분류기가 아니라
언어학적 분석을 위한 보조 도구로 활용된다.

### 데이터 개요

본 연구에서 사용된 데이터는 구어 발화 전사 자료로 구성된 내부 연구용 코퍼스이며,
문장 단위로 직접화행/간접화행 여부가 라벨링되어 있다.

데이터는 사전 라벨링된 한국어 발화 문장으로 구성된 엑셀 파일 형태로 제공되었다.

In [1]:
!pip -q install "transformers>=4.35" "accelerate>=0.25" pandas openpyxl scikit-learn torch

### 데이터 구성 및 라벨링

본 실험에서는 엑셀 파일에 정리된 발화 자료를 사용하며,
문장과 화행의 직접/간접 여부를 라벨로 활용한다.
직접화행은 0, 간접화행은 1로 매핑하였다.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

XLSX_PATH = "반존대 용례태깅_화행.xlsx"
SHEET = "반존대 용례 태깅본_단언강도_시작문장"

df = pd.read_excel(XLSX_PATH, sheet_name=SHEET)

#필요한 컬럼만 선택
df = df[["문장", "화행의직간접성"]].dropna()

df["화행의직간접성"] = df["화행의직간접성"].astype(str).str.strip()
df["문장"] = df["문장"].astype(str).str.strip()
df = df[df["화행의직간접성"].isin(["직접", "간접"])].copy()

label_map = {"직접": 0, "간접": 1}
df["label"] = df["화행의직간접성"].map(label_map)
df["text"] = df["문장"]

### 데이터 분할 전략

본 연구에서는 데이터를 train/validation/test(80/10/10)로 분할하였다.

간접화행의 비중이 낮기 때문에,
각 분할에서 간접화행이 소실되는 것을 방지하기 위해
stratified split을 조건부로 적용하였다.

In [3]:
counts = df["label"].value_counts()
print("전체 라벨 분포:\n", counts)

# stratify 가능 여부 판단
use_stratify = (counts.min() >= 3)
strat_col = df["label"] if use_stratify else None

train_df, temp_df = train_test_split(
    df[["text", "label"]],
    test_size=0.2,
    random_state=42,
    stratify=strat_col
)

temp_counts = temp_df["label"].value_counts()
use_stratify2 = (temp_counts.min() >= 2)
strat_col2 = temp_df["label"] if use_stratify2 else None

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=strat_col2
)

train_df.to_csv("train.csv", index=False)
valid_df.to_csv("valid.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("\ntrain 분포:\n", train_df["label"].value_counts())
print("valid 분포:\n", valid_df["label"].value_counts())
print("test  분포:\n", test_df["label"].value_counts())

전체 라벨 분포:
 label
0    973
1     21
Name: count, dtype: int64

train 분포:
 label
0    778
1     17
Name: count, dtype: int64
valid 분포:
 label
0    97
1     2
Name: count, dtype: int64
test  분포:
 label
0    98
1     2
Name: count, dtype: int64


In [4]:
import pandas as pd
import torch
from transformers import AutoTokenizer

MODEL_NAME = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("valid.csv")
test_df  = pd.read_csv("test.csv")

def encode_texts(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=128
    )

train_enc = encode_texts(train_df["text"].tolist())
valid_enc = encode_texts(valid_df["text"].tolist())
test_enc  = encode_texts(test_df["text"].tolist())

train_labels = train_df["label"].astype(int).tolist()
valid_labels = valid_df["label"].astype(int).tolist()
test_labels  = test_df["label"].astype(int).tolist()

class TextClsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TextClsDataset(train_enc, train_labels)
valid_dataset = TextClsDataset(valid_enc, valid_labels)
test_dataset  = TextClsDataset(test_enc,  test_labels)

print("train/valid/test size:", len(train_dataset), len(valid_dataset), len(test_dataset))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train/valid/test size: 795 99 100


### 클래스 불균형 처리 및 평가 지표

간접화행은 전체 데이터에서 소수 클래스이므로,
학습 시 클래스 가중치를 적용하여
간접화행 분류 오류에 더 큰 페널티를 부여하였다.

또한 단순 정확도 대신 macro-F1 점수를 주요 평가 지표로 활용하였다.

In [5]:
import pandas as pd
import numpy as np
import torch
from torch import nn

from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed
)

MODEL_NAME = "klue/roberta-base"

# 재현성을 위해 시드 고정
set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

counts = train_df["label"].value_counts().sort_index()
total = counts.sum()
weights = torch.tensor([total/(2*counts.get(0,1)), total/(2*counts.get(1,1))], dtype=torch.float)
print("class weights:", weights.tolist())

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = float((preds == labels).mean())

    tp = int(((preds == 1) & (labels == 1)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    recall_indirect = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0

    macro_f1 = f1_score(labels, preds, average="macro")

    return {
        "accuracy": acc,
        "recall_indirect(1)": recall_indirect,
        "macro_f1": macro_f1
    }

args = TrainingArguments(
    output_dir="./out_direct_indirect",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    logging_steps=50,
    report_to="none",
    save_strategy="no",
    seed=42,
)

trainer = WeightedTrainer(
    class_weights=weights,
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


class weights: [0.5109254717826843, 23.382352828979492]


/tmp/ipython-input-3085195273.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Step,Training Loss
50,0.602300
100,0.928600
150,0.808700
200,0.239400
250,0.102300


TrainOutput(global_step=250, training_loss=0.536265718460083, metrics={'train_runtime': 90.001, 'train_samples_per_second': 44.166, 'train_steps_per_second': 2.778, 'total_flos': 261466611264000.0, 'train_loss': 0.536265718460083, 'epoch': 5.0})

학습된 모델은 이후 간접화행 후보 탐색 및 후속 실험에 재사용하기 위해 저장한다.

In [6]:
trainer.save_model("model_direct_indirect")
tokenizer.save_pretrained("model_direct_indirect")

('model_direct_indirect/tokenizer_config.json',
 'model_direct_indirect/special_tokens_map.json',
 'model_direct_indirect/vocab.txt',
 'model_direct_indirect/added_tokens.json',
 'model_direct_indirect/tokenizer.json')

In [7]:
print("VALID 평가:")
print(trainer.evaluate(valid_dataset))

print("\nTEST 최종 평가:")
print(trainer.evaluate(test_dataset))

VALID 평가:


{'eval_loss': 0.7266187071800232, 'eval_accuracy': 0.98989898989899, 'eval_recall_indirect(1)': 0.5, 'eval_macro_f1': 0.8307692307692307, 'eval_runtime': 0.5369, 'eval_samples_per_second': 184.385, 'eval_steps_per_second': 7.45, 'epoch': 5.0}

TEST 최종 평가:
{'eval_loss': 0.6032083034515381, 'eval_accuracy': 0.98, 'eval_recall_indirect(1)': 0.5, 'eval_macro_f1': 0.7448979591836735, 'eval_runtime': 0.4908, 'eval_samples_per_second': 203.738, 'eval_steps_per_second': 8.15, 'epoch': 5.0}


### 실험 결과 해석

테스트 데이터에서 간접화행 재현율은 0.5로 나타났으며,
이는 간접화행 표본 수가 극히 제한된 조건에서
모델 성능의 변동성이 크다는 점을 보여준다.

따라서 본 결과를 일반적인 분류 성능으로 해석하는 데에는 한계가 있다.


한편 반문형과 같이 화행 기능이 비교적 명확한 간접화행은
정확히 분류된 반면,
조건절 기반의 장문 간접화행은 직접화행으로 오분류되었다.

이는 문장 단위 입력에 기반한 모델이
담화적 누적 효과에 의존하는 간접화행을 포착하는 데
구조적 한계를 가짐을 시사한다.

In [8]:
print("valid label counts:\n", valid_df["label"].value_counts())
print("test  label counts:\n", test_df["label"].value_counts())

valid label counts:
 label
0    97
1     2
Name: count, dtype: int64
test  label counts:
 label
0    98
1     2
Name: count, dtype: int64


macro-F1 점수는 소수 클래스인 간접화행의 분류 성능이
전체 성능 평가에 동일한 비중으로 반영되도록 하기 위한 지표이다.

In [9]:
from sklearn.metrics import confusion_matrix, classification_report

preds = trainer.predict(test_dataset).predictions
pred_labels = np.argmax(preds, axis=1)

print(confusion_matrix(test_df["label"].values, pred_labels))
print(classification_report(test_df["label"].values, pred_labels, digits=4))

[[97  1]
 [ 1  1]]
              precision    recall  f1-score   support

           0     0.9898    0.9898    0.9898        98
           1     0.5000    0.5000    0.5000         2

    accuracy                         0.9800       100
   macro avg     0.7449    0.7449    0.7449       100
weighted avg     0.9800    0.9800    0.9800       100



In [10]:
test_texts = test_df["text"].tolist()
y_true = test_df["label"].values
y_pred = pred_labels

indirect_idx = np.where(y_true == 1)[0]
for i in indirect_idx[:20]:
    print("문장:", test_texts[i])
    print("정답:", "간접", "예측:", "간접" if y_pred[i]==1 else "직접")
    print("---")

문장: 어떻게 안 해?
정답: 간접 예측: 간접
---
문장: 그 꼭 필요한 거라면 정말 내 생활 하는 데 있어서 그게 꼭 필요한 거라면은 하나쯤 장만해야 되지 않을까?
정답: 간접 예측: 직접
---


In [11]:
import os
import json
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    accuracy_score,
    recall_score
)

SEED = 42
RUN_NAME = f"seed_{SEED}"
SAVE_DIR = f"./artifacts/{RUN_NAME}"

os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

valid_metrics = trainer.evaluate(valid_dataset)
test_metrics  = trainer.evaluate(test_dataset)

metrics = {
    "seed": SEED,
    "valid": valid_metrics,
    "test": test_metrics
}

with open(f"{SAVE_DIR}/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

pred_output = trainer.predict(test_dataset)

logits = pred_output.predictions
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds = np.argmax(logits, axis=1)

y_true = test_df["label"].values
texts  = test_df["text"].values

pred_df = pd.DataFrame({
    "text": texts,
    "y_true": y_true,
    "y_pred": preds,
    "p_direct(0)": probs[:, 0],
    "p_indirect(1)": probs[:, 1]
})

pred_df.to_csv(
    f"{SAVE_DIR}/test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

cm = confusion_matrix(y_true, preds)

with open(f"{SAVE_DIR}/confusion_matrix.txt", "w", encoding="utf-8") as f:
    f.write(str(cm))

report = classification_report(
    y_true,
    preds,
    target_names=["direct(0)", "indirect(1)"],
    digits=4
)

with open(f"{SAVE_DIR}/classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

summary = {
    "accuracy": accuracy_score(y_true, preds),
    "macro_f1": f1_score(y_true, preds, average="macro"),
    "recall_indirect": recall_score(y_true, preds, pos_label=1)
}

with open(f"{SAVE_DIR}/summary_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

### 한계 및 향후 연구

본 실험은 간접화행 데이터의 수가 제한된 상황에서 수행되었다.

향후 규칙 기반 후보 추출과 반자동 라벨링을 통해
데이터를 확장하고,
간접화행의 하위 유형 분석으로 연구를 확장할 수 있을 것이다.